---

# C4. Exercițiu individual: construirea unui mini-prompt de adnotare

În acest exercițiu construiești un prompt mic de adnotare pentru comentarii politice.
- Intelegem cum se construiește un prompt: rol, variabile, definiții, reguli și format JSON.
- Alegemdouă axe proprii sau două axe din curs și vei testa promptul pe 5 comentarii.


## Pasul 0 . Configurare

In [1]:
import os, json, re, random
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
# caută .env urcând din folderul curent

ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

# DeepSeek
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
DEEPSEEK_MODEL = "deepseek-chat"
# Gemini prin OpenAI-compatible API
gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-2.5-flash-lite"
# alegem modelul pentru demo

USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Root project:", ROOT)
print("DeepSeek key:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)
print("Model folosit:", model_now)
print("OK")

Root project: c:\Users\User\Desktop\Analiza Datelor Complexe\AN2\curs_Inginerie_AI\proiect_AI\echochamber-project-team-4
DeepSeek key: True
Gemini key: True
Model folosit: gemini-2.5-flash-lite
OK


## Corpus

In [2]:
import pandas as pd
import random

corpus = pd.read_json("../../data/cleaned/corpus_youtube_sample.jsonl", lines=True)

print(len(corpus), "comentarii")
print("Câmpuri:", list(corpus.columns))

for _, c in corpus.sample(3).iterrows():
    print(f"[{c['source_channel'][:30]}] {c['text'][:80]}")

420 comentarii
Câmpuri: ['id', 'source_channel', 'video_title', 'text']
[georgesimionoficial] Păi ați refuzat dialogul la televiziune. Cum adică nu puteți să vă faceți campan
[@CălinGeorgescu-CanalulOficial] Nu există cuvinte de mulțumire pentru tot ce faceți pentru noi.Dumnezeu să vă oc
[georgesimionoficial] Abia după ce l-am votat mi-am dat seama că Simion creează impresia de apropiere 


### Pasul 1 Alege două axe

Alege două axe pe care vrei să le codezi.
Poți folosi axe din curs:
- institutional
- legitimare
- epistemic
- geopolitic
- mobilizare
Sau poți propune axe proprii:
- media_distrust
- elite_blame
- religious_frame
- fear
- irony
- people_vs_elite
- anti_corruption
- national_identity
Condiție: fiecare axă trebuie să aibă valori clare.
Pentru acest exercițiu folosim o scală simplă:
0 = absent
1 = prezent


In [3]:
# modifica dupa preferinte

AXA_1 = "religious_frame"
AXA_2 = "elite_blame"

## Pasul 2 — Definește axele
Scrie mai jos, în propriile cuvinte, ce înseamnă fiecare axă.
Exemplu:
media_distrust = comentariul exprimă neîncredere în presă, jurnaliști, televiziuni sau media mainstream.
religious_frame = comentariul folosește limbaj religios pentru a interpreta politica.

In [4]:
AXA_1_DEFINITION = """
religious_frame măsoară dacă textul foloseste religia ca justificare morala,
sau asociaza identitatea politica cu credinta prin valori religioase.
0 = absent
1 = prezent
2 = dominant
"""
AXA_2_DEFINITION = """
elite_blame măsoară dacă textul indica vina pe elire, construieste opozitia anti-sistem si sugereaza conspiratii.
0 = absent
1 = prezent
2 = dominant
"""

## Pasul 3 — Construiește mini-promptul
Promptul trebuie să conțină:
1. rolul modelului;
2. sarcina;
3. definițiile celor două axe;
4. regulile de codare;
5. formatul JSON.
Important:
- nu cere modelului să identifice direct „bula”;
- nu cere text liber;
- returnează doar JSON valid.

In [5]:
MINI_PROMPT = f"""
Ești speialist in analiza comentariilor politice de pe YouTube
SARCINĂ:
Adnotează comentariul folosind două axe:
1. {AXA_1}
2. {AXA_2}
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
{AXA_1} = 0 / 1 / 2
{AXA_2} = 0 / 1 / 2
DEFINIȚII:
{AXA_1_DEFINITION}
{AXA_2_DEFINITION}
REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
5. Pentru axe: 0 = absent, 1 = prezent, 2 = dominant.
6. Nu atribui direct o bulă discursivă.
7. Returnează doar JSON valid.
FORMAT OUTPUT:
{{
  "target": "",
  "stance": "",
  "tone": "",
  "{AXA_1}": 0,
  "{AXA_2}": 0
}}
"""
print(MINI_PROMPT)


Ești speialist in analiza comentariilor politice de pe YouTube
SARCINĂ:
Adnotează comentariul folosind două axe:
1. religious_frame
2. elite_blame
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
religious_frame = 0 / 1 / 2
elite_blame = 0 / 1 / 2
DEFINIȚII:

religious_frame măsoară dacă textul foloseste religia ca justificare morala,
sau asociaza identitatea politica cu credinta prin valori religioase.
0 = absent
1 = prezent
2 = dominant


elite_blame măsoară dacă textul indica vina pe elire, construieste opozitia anti-sistem si sugereaza conspiratii.
0 = absent
1 = prezent
2 = dominant

REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenți

## Pasul 4 — Alege 5 comentarii de test
Folosim un eșantion mic. Nu adnotăm tot corpusul.
Schimbă `random_state` ca să primești alte comentarii.

In [7]:
TESTS = corpus.sample(5, random_state=27)
TESTS[["id", "source_channel", "video_title", "text"]].head()

,id,source_channel,video_title,text
386,yt_iH8jB4NlV9Y_Ugz2SGyAmn4oGKy_iql4AaABAg,georgesimionoficial,Episodul 2: Cum ne-au furat alegerile - Turism...,Violul de la tabara AUR nu este o minciuna! Ex...
95,yt_bnbjgKIS5Dw_UgxL_T_1GYor2ca7aW14AaABAg,DianaSosoacaOfficial,Mesaj pentru Popi! - Euparlamentar @DianaSosoa...,Cine mai poate să se încreadă în asemenea indi...
82,yt_DKhN-ua4lyw_UgzarZ9xZqzHNMf7H8l4AaABAg,georgesimionoficial,#gs #georgesimion #democratie #impreuna #prosp...,Respect d.nul George Simion ! Am o singura int...
156,yt_yEuctxNb4O0_Ugw9nyUkmTKMRiHEUA54AaABAg,NicusorDanRO,🟢 LIVE - Întâlnire la Palatul Cotroceni cu mag...,"O dezbatere unicat in Romania, respect tuturor..."
21,yt_PI9K4fsNHvg_UgwZ3VabzgKdxQi62W54AaABAg,StirileProTV,Donald Trump amenință aliații din NATO pentru ...,Sufletul Pământului în rezervorul şi motorul b...


## Pasul 5 — Rulează promptul pe cele 5 comentarii
Pentru fiecare comentariu:
1. trimitem canalul, titlul video și textul;
2. modelul returnează JSON;
3. citim rezultatul și verificăm dacă are sens.

In [8]:
USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Using:", model_now)

Using: gemini-2.5-flash-lite


In [9]:
def llm(system, user, max_tokens=700):
    response = client_now.chat.completions.create(
        model=model_now,
        temperature=0,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return response.choices[0].message.content

In [10]:
results = []
for _, row in TESTS.iterrows():
    USER = f"""
CANAL:
{row.get("source_channel", "")}
TITLU VIDEO:
{row.get("video_title", "")}
COMENTARIU:
<<< {row["text"]} >>>
"""
    raw = llm(MINI_PROMPT, USER, max_tokens=300)
    print("=" * 80)
    print("COMENTARIU:")
    print(row["text"])
    print()
    print("OUTPUT MODEL:")
    print(raw)
    results.append({
        "id": row["id"],
        "text": row["text"],
        "model_output": raw
    })

COMENTARIU:
Violul de la tabara AUR nu este o minciuna! Exista raport medico-legal care a demonstrat faptul ca fata a fost victima unui viol. Sa nu recunosti intamplarea si sa o numesti minciuna arata doar cat de interesat e gunoiul asta de siguranta femeilor

OUTPUT MODEL:
```json
{
  "target": "AUR",
  "stance": "anti",
  "tone": "acuzator",
  "religious_frame": 0,
  "elite_blame": 0
}
```
COMENTARIU:
Cine mai poate să se încreadă în asemenea indivizi care și-au vândut sufletul pentru o pungă le arginți cine vrea cu adevărat să creadă în Dumnezeu îl poate regăsi înăuntrul său nu vă trebuie nicio locație ❤

OUTPUT MODEL:
```json
{
  "target": "Diana Sosoaca",
  "stance": "pro",
  "tone": "affective",
  "religious_frame": 1,
  "elite_blame": 0
}
```
COMENTARIU:
Respect d.nul George Simion ! Am o singura intrebare , de ce va numeste Marius Lulea " arogant " , de fapt da impresia ca va sapă ? La Cristache in emisiune, sunt şocată....

OUTPUT MODEL:
```json
{
  "target": "George Simion",


In [ ]:
## Pasul 6 — Interpretare scurtă
Completează în notebook, în 3–5 rânduri:
- Ce două axe ai ales?
- De ce le-ai ales?
- Modelul a returnat JSON corect?
- Care a fost cea mai mare problemă?
- Ce ai schimba în prompt?

### Am ales axele: Religious Frame si Elite Blame, studiind bula anti-sistem
### Am considerat ca aceste doua axe ar caracteriza bine bula tinta, intrucat o persoana critica de elita poate avea o retorica anti-sistem
### Factorul religios l-am ales din curiozitate, pentru a vedea daca exista o suprapunere dintre cele doua axe, in contextul politic din perspectiva anti-sistem
### Modelul json este corect, dupa formatarea indicata de prompt
### Aceste doua axe par a se suprapune limitat, dar in acelasi timp, comentariile alese nu prea descriu bine axele, dar exista rezultate satisfacatoare pentru cateva exemple. Putem deduce ca cele doua tipologii fac parte din bula anti-sistem, dar nu descriu in totalitate fenomenul.
### As alege alte axe/mai multe si cu siguranta mai multe comentarii pentru testare